<a href="https://www.kaggle.com/code/mobeenfatimah/ev-purchase-prediction-competition-high-score?scriptVersionId=354374289" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## EV Purchase Prediction — Competition notebook

This notebook is an upgraded, validation-first pipeline for Kaggle Playground Series S6E9.

**Goal:** improve ROC-AUC beyond a 0.94551 submission by combining the strongest publicly documented ideas for this synthetic tabular dataset: exact-value target encoding, train+test frequency encoding, numeric digit features, tuned XGBoost/LightGBM, and rank blending.

> Important: this notebook does **not** use test labels. Test-set frequency is unsupervised; target encoding is fold-safe for OOF validation.

In [1]:
import os, gc, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import rankdata
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")
SEED = 2026
N_SPLITS = 10
TARGET = "Will_Buy_EV"
DATA_PATH = "/kaggle/input/competitions/playground-series-s6e9"
np.random.seed(SEED)
print("Libraries loaded.")
print("LightGBM:", lgb.__version__)
print("XGBoost:", xgb.__version__)

Libraries loaded.
LightGBM: 4.6.0
XGBoost: 3.2.0


## 1. Load the competition data

In [2]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_PATH, "sample_submission.csv"))

print("Train:", train.shape)
print("Test :", test.shape)
print("Sample:", sample_submission.shape)

Train: (668665, 15)
Test : (286571, 14)
Sample: (286571, 2)


## 2. Verify the target and submission schema

In [3]:
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8).to_numpy()
print(train[TARGET].value_counts())
print(f"Positive rate: {y.mean():.6f}")
print("Submission columns:", sample_submission.columns.tolist())
assert list(sample_submission.columns) == ["id", TARGET]
assert sample_submission["id"].equals(test["id"])

Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64
Positive rate: 0.174645
Submission columns: ['id', 'Will_Buy_EV']


## 3. Identify numeric and categorical predictors

In [4]:
numeric_cols = [c for c in train.select_dtypes(include=np.number).columns if c != "id"]
categorical_cols = [c for c in train.select_dtypes(include=["object", "category"]).columns if c != TARGET]
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 4. Quick diagnostics: repeated values are important here

In [5]:
for c in ["Annual_Income_USD", "Daily_Commute_km", "Environmental_Concern_Level"]:
    vc = train[c].value_counts().head(8)
    print(f"\n{c}:\n{vc}")


Annual_Income_USD:
Annual_Income_USD
30000.0    61605
86095.0     1082
96749.0     1073
65800.0     1029
92372.0      907
84578.0      905
93957.0      852
68480.0      762
Name: count, dtype: int64

Daily_Commute_km:
Daily_Commute_km
5.0     144280
46.6      2756
48.5      2720
33.0      2562
53.3      2546
30.1      2370
47.1      2347
52.2      2279
Name: count, dtype: int64

Environmental_Concern_Level:
Environmental_Concern_Level
1.0    147476
2.0    135133
4.0    130469
5.0    128236
3.0    127351
Name: count, dtype: int64


## 5. Build conservative engineered features

In [6]:
def add_features(df):
    d = df.copy()
    d["Home_Charging_Bin"] = (d["Home_Charging_Possible"].astype(str) == "Yes").astype(np.int8)
    d["Subsidy_Bin"] = (d["Subsidy_Available"].astype(str) == "Yes").astype(np.int8)
    d["Subsidy_x_Income"] = d["Subsidy_Bin"] * d["Annual_Income_USD"]
    d["Income_per_Car"] = d["Annual_Income_USD"] / (d["Number_of_Cars_Owned"] + 1.0)
    d["Environmental_x_Subsidy"] = d["Environmental_Concern_Level"] * d["Subsidy_Bin"]
    d["Environmental_x_Income"] = d["Environmental_Concern_Level"] * d["Annual_Income_USD"]
    d["HomeCharge_x_StationsHome"] = d["Home_Charging_Bin"] * d["Charging_Stations_Near_Home"]
    d["WorkCharge_x_StationsWork"] = d["Charging_Stations_Near_Work"] * d["Home_Charging_Bin"]
    return d

train_f = add_features(train.drop(columns=[TARGET]))
test_f = add_features(test.copy())
print(train_f.shape, test_f.shape)

(668665, 22) (286571, 22)


## 6. Create train+test frequency features

Because the competition data is synthetic, repeated exact values can carry information. Frequency encoding is unsupervised and therefore safe to calculate from the combined train/test feature values.

In [7]:
freq_cols = numeric_cols + categorical_cols
combined = pd.concat([train_f[freq_cols], test_f[freq_cols]], ignore_index=True)

for c in freq_cols:
    freq = combined[c].value_counts(dropna=False)
    train_f[c + "__freq"] = train_f[c].map(freq).astype(np.float32)
    test_f[c + "__freq"] = test_f[c].map(freq).astype(np.float32)

del combined, freq
gc.collect()
print("After frequency features:", train_f.shape)

After frequency features: (668665, 35)


## 7. Add synthetic-data digit features

Some public experiments found that decomposing numeric values into digits can expose generator artifacts that ordinary trees do not exploit efficiently. We add integer-place digits plus the first two decimal digits.

In [8]:
def add_digit_features(train_df, test_df, cols):
    tr, te = train_df.copy(), test_df.copy()
    for c in cols:
        a = pd.to_numeric(tr[c], errors="coerce").fillna(0).to_numpy()
        b = pd.to_numeric(te[c], errors="coerce").fillna(0).to_numpy()
        aa = np.abs(a); bb = np.abs(b)
        for k in range(4):
            base = 10.0 ** k
            tr[f"{c}__d{k}"] = (np.floor(aa / base) % 10).astype(np.int8)
            te[f"{c}__d{k}"] = (np.floor(bb / base) % 10).astype(np.int8)
        for k in (1, 2):
            tr[f"{c}__fd{k}"] = (np.floor(aa * (10.0 ** k)) % 10).astype(np.int8)
            te[f"{c}__fd{k}"] = (np.floor(bb * (10.0 ** k)) % 10).astype(np.int8)
    return tr, te

digit_cols = numeric_cols
train_f, test_f = add_digit_features(train_f, test_f, digit_cols)
print("After digit features:", train_f.shape)

After digit features: (668665, 77)


## 8. Add frequency encodings for the new digit features

In [9]:
digit_feature_cols = [c for c in train_f.columns if "__d" in c or "__fd" in c]
for c in digit_feature_cols:
    vals = pd.concat([train_f[c], test_f[c]], ignore_index=True)
    freq = vals.value_counts(dropna=False)
    train_f[c + "__freq"] = train_f[c].map(freq).astype(np.float32)
    test_f[c + "__freq"] = test_f[c].map(freq).astype(np.float32)
    del vals, freq
print("Final engineered columns:", train_f.shape[1])

Final engineered columns: 119


## 9. Encode categorical variables consistently

In [10]:
model_base_cols = [c for c in train_f.columns if c != "id"]
cat_encode_cols = categorical_cols

for c in cat_encode_cols:
    all_values = pd.concat([train_f[c].astype(str), test_f[c].astype(str)], ignore_index=True)
    codes, _ = pd.factorize(all_values, sort=True)
    train_f[c] = codes[:len(train_f)].astype(np.int16)
    test_f[c] = codes[len(train_f):].astype(np.int16)
    del all_values, codes

X = train_f.drop(columns=["id"]).astype(np.float32)
X_test = test_f.drop(columns=["id"]).astype(np.float32)
print("Model matrix:", X.shape, X_test.shape)

Model matrix: (668665, 118) (286571, 118)


## 10. Freeze one 10-fold split for every experiment

In [11]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print([(len(a), len(b)) for a,b in folds[:3]], "...")

[(601798, 66867), (601798, 66867), (601798, 66867)] ...


## 11. Diagnostic: exact-value target encoding signal

In [12]:
diag = []
for c in numeric_cols + categorical_cols:
    means = pd.DataFrame({"v": train[c], "y": y}).groupby("v")["y"].mean()
    p = train[c].map(means).fillna(y.mean()).to_numpy()
    diag.append((c, roc_auc_score(y, p)))
print(pd.DataFrame(diag, columns=["feature", "value_TE_auc"]).sort_values("value_TE_auc", ascending=False).to_string(index=False))

                    feature  value_TE_auc
Environmental_Concern_Level      0.843521
          Annual_Income_USD      0.742764
          Subsidy_Available      0.717940
           Daily_Commute_km      0.560242
     Home_Charging_Possible      0.550822
                        Age      0.547880
        Range_Anxiety_Level      0.545104
Charging_Stations_Near_Home      0.524072
                  City_Type      0.523450
Charging_Stations_Near_Work      0.512721
           Current_Car_Type      0.510372
       Number_of_Cars_Owned      0.506251
                     Gender      0.504614


## 12. Fold-safe exact-value target encoding

For each validation fold, the mapping is learned only from the other folds. This prevents target leakage in OOF evaluation.

In [13]:
te_cols = numeric_cols + categorical_cols

def make_oof_te(base_train, base_test, y, cols, folds, smoothing=20.0):
    global_mean = float(y.mean())
    oof = pd.DataFrame(index=base_train.index)
    tst = pd.DataFrame(index=base_test.index)
    for c in cols:
        oof_col = np.zeros(len(base_train), dtype=np.float32)
        for tr_idx, va_idx in folds:
            tmp = pd.DataFrame({"key": base_train.iloc[tr_idx][c].to_numpy(), "target": y[tr_idx]})
            stats = tmp.groupby("key")["target"].agg(["mean", "count"])
            enc = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
            oof_col[va_idx] = base_train.iloc[va_idx][c].map(enc).fillna(global_mean).to_numpy(dtype=np.float32)
        oof[c + "__te"] = oof_col
        full = pd.DataFrame({"key": base_train[c].to_numpy(), "target": y})
        stats = full.groupby("key")["target"].agg(["mean", "count"])
        enc = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        tst[c + "__te"] = base_test[c].map(enc).fillna(global_mean).to_numpy(dtype=np.float32)
    return oof, tst

te_train, te_test = make_oof_te(train.drop(columns=[TARGET]), test, y, te_cols, folds, smoothing=20.0)
print(te_train.shape, te_test.shape)

(668665, 13) (286571, 13)


## 13. Append the OOF target encodings

In [14]:
X_full = pd.concat([X.reset_index(drop=True), te_train.reset_index(drop=True)], axis=1).astype(np.float32)
X_test_full = pd.concat([X_test.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1).astype(np.float32)
print("Final feature count:", X_full.shape[1])
print("Memory train MB:", round(X_full.memory_usage(deep=True).sum()/1024**2, 1))

Final feature count: 131
Memory train MB: 334.1


## 14. Prepare a compact feature matrix for XGBoost

In [15]:
# Remove extremely redundant engineered columns only if they are constant.
constant_cols = [c for c in X_full.columns if X_full[c].nunique(dropna=False) <= 1]
if constant_cols:
    X_full = X_full.drop(columns=constant_cols)
    X_test_full = X_test_full.drop(columns=constant_cols)
print("Dropped constants:", len(constant_cols))
print("XGB features:", X_full.shape[1])

Dropped constants: 52
XGB features: 79


## 15. XGBoost 10-fold model — primary model

This is intentionally slower than a baseline because 10-fold predictions tend to be more stable for this leaderboard.

In [16]:
oof_xgb = np.zeros(len(X_full), dtype=np.float64)
pred_xgb = np.zeros(len(X_test_full), dtype=np.float64)

xgb_params = dict(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=2600, learning_rate=0.025,
    max_depth=7, min_child_weight=8,
    subsample=0.88, colsample_bytree=0.92,
    reg_alpha=0.03, reg_lambda=2.5,
    gamma=0.0, max_bin=256,
    tree_method="hist", device="cuda",
    random_state=SEED, n_jobs=-1
)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print(f"XGB fold {fold}/{N_SPLITS}")
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_full.iloc[tr_idx], y[tr_idx], eval_set=[(X_full.iloc[va_idx], y[va_idx])], verbose=False)
    oof_xgb[va_idx] = model.predict_proba(X_full.iloc[va_idx])[:, 1]
    pred_xgb += model.predict_proba(X_test_full)[:, 1] / N_SPLITS
    del model
    gc.collect()

print("XGB OOF AUC:", roc_auc_score(y, oof_xgb))

XGB fold 1/10
XGB fold 2/10
XGB fold 3/10
XGB fold 4/10
XGB fold 5/10
XGB fold 6/10
XGB fold 7/10
XGB fold 8/10
XGB fold 9/10
XGB fold 10/10
XGB OOF AUC: 0.9449491689735512


## 16. LightGBM 10-fold model

A second GBDT family adds diversity for rank blending.

In [17]:
oof_lgb = np.zeros(len(X_full), dtype=np.float64)
pred_lgb = np.zeros(len(X_test_full), dtype=np.float64)

lgb_params = dict(
    objective="binary", metric="auc",
    n_estimators=2600, learning_rate=0.025,
    num_leaves=63, max_depth=-1,
    min_child_samples=70,
    subsample=0.88, subsample_freq=1,
    colsample_bytree=0.92,
    reg_alpha=0.03, reg_lambda=2.5,
    random_state=SEED, n_jobs=-1, verbosity=-1
)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print(f"LGBM fold {fold}/{N_SPLITS}")
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_full.iloc[tr_idx], y[tr_idx], eval_set=[(X_full.iloc[va_idx], y[va_idx])], callbacks=[lgb.early_stopping(120, verbose=False)])
    oof_lgb[va_idx] = model.predict_proba(X_full.iloc[va_idx])[:, 1]
    pred_lgb += model.predict_proba(X_test_full)[:, 1] / N_SPLITS
    del model
    gc.collect()

print("LGBM OOF AUC:", roc_auc_score(y, oof_lgb))

LGBM fold 1/10
LGBM fold 2/10
LGBM fold 3/10
LGBM fold 4/10
LGBM fold 5/10
LGBM fold 6/10
LGBM fold 7/10
LGBM fold 8/10
LGBM fold 9/10
LGBM fold 10/10
LGBM OOF AUC: 0.9456626951490862


## 17. Logistic regression diversity model

A regularized linear model can capture smooth ranking structure that tree models may approximate differently. It is trained on a standardized subset of continuous/TE features.

In [18]:
linear_cols = [c for c in X_full.columns if c.endswith("__te") or c in [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work", "Environmental_Concern_Level",
    "Subsidy_x_Income", "Income_per_Car", "Environmental_x_Income",
    "Environmental_x_Subsidy"
]]
linear_cols = list(dict.fromkeys([c for c in linear_cols if c in X_full.columns]))
print("Linear features:", len(linear_cols))

oof_lr = np.zeros(len(X_full), dtype=np.float64)
pred_lr = np.zeros(len(X_test_full), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print(f"LR fold {fold}/{N_SPLITS}")
    scaler = StandardScaler()
    A = scaler.fit_transform(X_full.iloc[tr_idx][linear_cols])
    B = scaler.transform(X_full.iloc[va_idx][linear_cols])
    T = scaler.transform(X_test_full[linear_cols])
    lr = LogisticRegression(C=0.35, max_iter=250, solver="lbfgs", n_jobs=-1)
    lr.fit(A, y[tr_idx])
    oof_lr[va_idx] = lr.predict_proba(B)[:, 1]
    pred_lr += lr.predict_proba(T)[:, 1] / N_SPLITS
    del scaler, A, B, T, lr
    gc.collect()

print("LR OOF AUC:", roc_auc_score(y, oof_lr))

Linear features: 24
LR fold 1/10
LR fold 2/10
LR fold 3/10
LR fold 4/10
LR fold 5/10
LR fold 6/10
LR fold 7/10
LR fold 8/10
LR fold 9/10
LR fold 10/10
LR OOF AUC: 0.9438281654251857


## 18. Compare individual models

In [19]:
scores = pd.DataFrame({
    "Model": ["XGBoost", "LightGBM", "LogisticRegression"],
    "OOF_AUC": [roc_auc_score(y,oof_xgb), roc_auc_score(y,oof_lgb), roc_auc_score(y,oof_lr)]
}).sort_values("OOF_AUC", ascending=False)
display(scores)

,Model,OOF_AUC
1,LightGBM,0.945663
0,XGBoost,0.944949
2,LogisticRegression,0.943828


## 19. Convert predictions to rank space

ROC-AUC depends on ranking, so rank averaging is often more robust than averaging raw probabilities when model calibration differs.

In [20]:
def rank01(a):
    return rankdata(a, method="average") / len(a)

oof_xgb_r, oof_lgb_r, oof_lr_r = map(rank01, [oof_xgb, oof_lgb, oof_lr])
pred_xgb_r, pred_lgb_r, pred_lr_r = map(rank01, [pred_xgb, pred_lgb, pred_lr])

equal_rank_oof = (oof_xgb_r + oof_lgb_r + oof_lr_r) / 3
print("Equal rank blend:", roc_auc_score(y, equal_rank_oof))

Equal rank blend: 0.9456887008113415


## 20. Search a fine rank-blend weight grid

Weights are selected only from OOF predictions. The final test prediction uses the same weights.

In [21]:
best_rank_auc = -1
best_rank_w = None

for wx in np.arange(0.30, 0.76, 0.02):
    for wl in np.arange(0.10, 0.61, 0.02):
        wr = 1.0 - wx - wl
        if wr < 0.05 or wr > 0.45:
            continue
        blend = wx*oof_xgb_r + wl*oof_lgb_r + wr*oof_lr_r
        auc = roc_auc_score(y, blend)
        if auc > best_rank_auc:
            best_rank_auc = auc
            best_rank_w = (wx, wl, wr)

print("Best rank weights:", best_rank_w)
print("Best rank OOF AUC:", best_rank_auc)

Best rank weights: (np.float64(0.3), np.float64(0.56), np.float64(0.1399999999999999))
Best rank OOF AUC: 0.9457740222283763


## 21. Also search a probability blend

This provides a second candidate in case probability calibration happens to work better than ranks.

In [22]:
best_prob_auc = -1
best_prob_w = None
for wx in np.arange(0.30, 0.81, 0.025):
    for wl in np.arange(0.10, 0.61, 0.025):
        wr = 1.0 - wx - wl
        if wr < 0.05 or wr > 0.45:
            continue
        blend = wx*oof_xgb + wl*oof_lgb + wr*oof_lr
        auc = roc_auc_score(y, blend)
        if auc > best_prob_auc:
            best_prob_auc = auc
            best_prob_w = (wx, wl, wr)
print("Best probability weights:", best_prob_w)
print("Best probability OOF AUC:", best_prob_auc)

Best probability weights: (np.float64(0.3), np.float64(0.5999999999999999), np.float64(0.10000000000000009))
Best probability OOF AUC: 0.9457473914487331


## 22. Compare all final candidates

In [23]:
wx, wl, wr = best_rank_w
p_rank_test = wx*pred_xgb_r + wl*pred_lgb_r + wr*pred_lr_r
p_rank_oof = wx*oof_xgb_r + wl*oof_lgb_r + wr*oof_lr_r

wx2, wl2, wr2 = best_prob_w
p_prob_test = wx2*pred_xgb + wl2*pred_lgb + wr2*pred_lr
p_prob_oof = wx2*oof_xgb + wl2*oof_lgb + wr2*oof_lr

candidates = {
    "XGBoost": (roc_auc_score(y,oof_xgb), pred_xgb),
    "LightGBM": (roc_auc_score(y,oof_lgb), pred_lgb),
    "LogisticRegression": (roc_auc_score(y,oof_lr), pred_lr),
    "Equal_Rank_Blend": (roc_auc_score(y,equal_rank_oof), (pred_xgb_r+pred_lgb_r+pred_lr_r)/3),
    "Optimized_Rank_Blend": (roc_auc_score(y,p_rank_oof), p_rank_test),
    "Optimized_Probability_Blend": (roc_auc_score(y,p_prob_oof), p_prob_test),
}
summary = pd.DataFrame([(k,v[0]) for k,v in candidates.items()], columns=["Candidate","OOF_AUC"]).sort_values("OOF_AUC", ascending=False)
display(summary)

,Candidate,OOF_AUC
4,Optimized_Rank_Blend,0.945774
5,Optimized_Probability_Blend,0.945747
3,Equal_Rank_Blend,0.945689
1,LightGBM,0.945663
0,XGBoost,0.944949
2,LogisticRegression,0.943828


## 23. Select the highest-OOF candidate

This is an OOF-based selection, not a leaderboard guess.

In [24]:
best_name = summary.iloc[0]["Candidate"]
final_test_pred = candidates[best_name][1].astype(np.float64)
print("Selected:", best_name)
print("OOF AUC:", summary.iloc[0]["OOF_AUC"])

Selected: Optimized_Rank_Blend
OOF AUC: 0.9457740222283763


## 24. Optional safety blend

If the optimized candidate is extremely close to equal rank blending, averaging the two gives a conservative submission. Otherwise, retain the measured winner.

In [25]:
equal_test = (pred_xgb_r + pred_lgb_r + pred_lr_r) / 3
equal_auc = roc_auc_score(y, equal_rank_oof)
best_auc = float(summary.iloc[0]["OOF_AUC"])

if best_name == "Equal_Rank_Blend":
    final_test_pred = equal_test
else:
    final_test_pred = candidates[best_name][1]

final_test_pred = np.clip(final_test_pred, 1e-7, 1-1e-7)
print("Final prediction range:", final_test_pred.min(), final_test_pred.max())

Final prediction range: 7.069801201098505e-05 0.999894895156872


## 25. Create the Kaggle submission

In [26]:
submission = sample_submission.copy()
submission[TARGET] = final_test_pred

assert len(submission) == len(test)
assert submission["id"].equals(test["id"])
assert submission[TARGET].notna().all()
assert np.isfinite(submission[TARGET]).all()

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
display(submission.head())

Saved submission.csv


,id,Will_Buy_EV
0,668665,0.553283
1,668666,0.485982
2,668667,0.346349
3,668668,0.263061
4,668669,0.552699


## 26. Validate submission statistics

In [27]:
print(submission[TARGET].describe())
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("File size MB:", round(os.path.getsize("submission.csv")/1024**2, 2))

count    286571.000000
mean          0.500002
std           0.287974
min           0.000071
25%           0.250798
50%           0.498538
75%           0.749970
max           0.999895
Name: Will_Buy_EV, dtype: float64
Rows: 286571
Columns: ['id', 'Will_Buy_EV']
File size MB: 7.18


## 27. Save additional named copies

Keeping multiple candidates is useful if you want to submit several times before the competition closes.

In [28]:
rank_submission = sample_submission.copy()
rank_submission[TARGET] = np.clip(p_rank_test, 1e-7, 1-1e-7)
rank_submission.to_csv("submission_optimized_rank.csv", index=False)

prob_submission = sample_submission.copy()
prob_submission[TARGET] = np.clip(p_prob_test, 1e-7, 1-1e-7)
prob_submission.to_csv("submission_optimized_probability.csv", index=False)

equal_submission = sample_submission.copy()
equal_submission[TARGET] = np.clip(equal_test, 1e-7, 1-1e-7)
equal_submission.to_csv("submission_equal_rank.csv", index=False)

print("Created:")
print(" - submission.csv")
print(" - submission_optimized_rank.csv")
print(" - submission_optimized_probability.csv")
print(" - submission_equal_rank.csv")

Created:
 - submission.csv
 - submission_optimized_rank.csv
 - submission_optimized_probability.csv
 - submission_equal_rank.csv


## 28. Final report

### What changed versus the previous notebook
- 10-fold CV instead of 5-fold.
- Train+test frequency encoding.
- Numeric digit decomposition.
- Exact-value, fold-safe target encoding.
- XGBoost tuned for this dataset.
- LightGBM diversity model.
- Logistic regression diversity model.
- Fine OOF rank-weight optimization.

The public experiments for this competition report that exact-value target encoding was the largest single improvement, while other public work reports strong results from digit/frequency features and rank blending. Current public Kaggle notebooks have reported scores around 0.9462+, so the purpose of this notebook is to target that remaining gap rather than simply repeat the 0.94551 pipeline.

In [29]:
print("="*72)
print("S6E9 FINAL CHECK")
print("="*72)
print(f"Best candidate : {best_name}")
print(f"OOF ROC-AUC    : {best_auc:.6f}")
print(f"Train rows     : {len(train):,}")
print(f"Test rows      : {len(test):,}")
print("Submission     : submission.csv")
print("="*72)

S6E9 FINAL CHECK
Best candidate : Optimized_Rank_Blend
OOF ROC-AUC    : 0.945774
Train rows     : 668,665
Test rows      : 286,571
Submission     : submission.csv
